# Chunking Strategy

Validates the chunking approach for `ingestion/index.py` against the actual structure of the FE 501s repair manual.

## Strategy

Chunks follow a three-level hierarchy driven by the manual's own structure:

```
Chapter  (e.g. 7 — Handlebar, controls)
└── Section  (e.g. 7.2 — Adjusting the handlebar position)
    └── Phase  (Preparatory work | Main work | Reworking)
```

**Why section-level, not page-level or fixed-token?**  
Sections map directly to discrete procedures — removing a wheel, adjusting valve clearance, etc. Keeping a section together preserves procedural coherence so the agent gets a complete, self-contained chunk when it retrieves a step.

**Why split within sections by phase?**  
Some sections contain both a removal and installation procedure (or prep + execution + follow-up) under a single section number. Splitting on the manual's own phase headers (`Preparatory work`, `Main work`, `Reworking`) keeps chunks focused without inventing arbitrary token windows.

**Token cap fallback**  
If a single phase block exceeds the token budget, it is split at paragraph boundaries. In practice this rarely triggers — most phase blocks are 200–600 tokens.

In [1]:
import sys
sys.path.insert(0, "..")

from ingestion.parse import parse_manual, parse_toc
import pypdf
from collections import defaultdict

PDF = "../manuals/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du.pdf"

pages = parse_manual(PDF, extract_images=False)
toc   = parse_toc(pypdf.PdfReader(PDF))

print(f"Pages: {len(pages)}  |  TOC entries: {len(toc)}")

Pages: 378  |  TOC entries: 254


## 1. Phase header consistency

Verify how consistently `Preparatory work`, `Main work`, and `Reworking` appear across the manual, and whether any other structural headers need to be accounted for.

In [2]:
PHASE_HEADERS = ["Preparatory work", "Main work", "Reworking"]

for header in PHASE_HEADERS:
    count = sum(1 for p in pages if header in p.text)
    print(f"{header:20s}: {count} pages")

# Check whether any phases appear out of order (Main before Preparatory, etc.)
print()
out_of_order = 0
for p in pages:
    text = p.text
    positions = {h: text.find(h) for h in PHASE_HEADERS if h in text}
    if len(positions) >= 2:
        ordered = sorted(positions, key=lambda h: positions[h])
        expected = [h for h in PHASE_HEADERS if h in positions]
        if ordered != expected:
            out_of_order += 1
            print(f"  Out-of-order on page {p.page_num}: {ordered}")

if out_of_order == 0:
    print("All phase headers appear in correct order (Prep → Main → Rework) on every page.")

Preparatory work    : 76 pages
Main work           : 97 pages
Reworking           : 74 pages

  Out-of-order on page 20: ['Reworking', 'Preparatory work', 'Main work']
  Out-of-order on page 46: ['Reworking', 'Preparatory work', 'Main work']
  Out-of-order on page 65: ['Reworking', 'Preparatory work', 'Main work']
  Out-of-order on page 90: ['Main work', 'Reworking', 'Preparatory work']
  Out-of-order on page 92: ['Reworking', 'Preparatory work']
  Out-of-order on page 103: ['Main work', 'Reworking', 'Preparatory work']
  Out-of-order on page 104: ['Reworking', 'Preparatory work', 'Main work']
  Out-of-order on page 108: ['Reworking', 'Preparatory work']
  Out-of-order on page 135: ['Reworking', 'Preparatory work']
  Out-of-order on page 136: ['Main work', 'Reworking', 'Preparatory work']
  Out-of-order on page 137: ['Main work', 'Reworking', 'Preparatory work']
  Out-of-order on page 150: ['Main work', 'Reworking', 'Preparatory work']
  Out-of-order on page 172: ['Main work', 'Reworki

## 2. Section-level phase distribution

How many sections have multiple phases that warrant intra-section splitting?

In [3]:
# Group pages by section number
sections = defaultdict(list)
for p in pages:
    if p.section and p.chapter_num:  # skip front/back matter
        sections[p.section].append(p)

phase_counts = defaultdict(int)
examples = defaultdict(list)

for sec, sec_pages in sections.items():
    combined = "\n".join(p.text for p in sec_pages)
    n_phases = sum(1 for h in PHASE_HEADERS if h in combined)
    phase_counts[n_phases] += 1
    if n_phases >= 2:
        examples[n_phases].append((sec, sec_pages[0].chapter_title, sec_pages[0].page_num))

print(f"{'Phases':>8}  {'Sections':>10}  {'Notes'}")
print(f"{'0':>8}  {phase_counts[0]:>10}  no phase markers (short ref sections, spec tables)")
print(f"{'1':>8}  {phase_counts[1]:>10}  single-phase (no split needed)")
print(f"{'2':>8}  {phase_counts[2]:>10}  two-phase split")
print(f"{'3':>8}  {phase_counts[3]:>10}  full three-phase split")
print(f"{'Total':>8}  {len(sections):>10}")

print(f"\nSections needing intra-section splits: {phase_counts[2] + phase_counts[3]} "
      f"({100*(phase_counts[2]+phase_counts[3])/len(sections):.0f}% of sections)")

  Phases    Sections  Notes
       0         114  no phase markers (short ref sections, spec tables)
       1          37  single-phase (no split needed)
       2          42  two-phase split
       3          30  full three-phase split
   Total         223

Sections needing intra-section splits: 72 (32% of sections)


In [4]:
# Show a concrete example of a 3-phase section
sec, ch_title, start_pg = examples[3][2]  # pick a non-TOC example
sec_pages = sections[sec]
combined  = "\n".join(p.text for p in sec_pages)

print(f"Section {sec} — {ch_title}  (starts p.{start_pg})")
print("=" * 60)
print(combined[:1500])

Section 6.6 — Fork, triple clamp  (starts p.20)
6 Fork, triple clamp
20
Reworking
‒ Install the fork protector.
 (p. 22)
‒ Remove the motorcycle from the lift stand.
 (p. 15)
6.5 Removing the fork legs
Preparatory work
‒ Raise the motorcycle with a lift stand.
 (p. 15)
‒ Remove the front wheel.
 (p. 127)
‒ Remove the headlight mask with the headlight.
 (p. 121)
Main work
S04825-10
‒ Remove screw1 and take off the clamp.
‒ Remove the cable ties.
‒ Remove screws2 and take off the brake caliper.
‒ Allow the brake caliper and the brake line to hang loosely to
the side.
S05734-10
‒ Loosen screws3.
‒ Remove the left fork leg.
‒ Loosen screws4.
‒ Remove the right fork leg.
6.6 Installing the fork legs
Main work
W00297-11
‒ Position the fork legs.
✓ Bleeder screws1 are positioned toward the front.
Note
Grooves are milled into the side of the upper end of the
fork legs. The second milled groove (from the top) must
be flush with the upper edge of the upper triple clamp.
The pressure and rebound 

## 3. Chunk size estimate

Estimate the token count per chunk to validate the token cap assumption. Using the rough approximation of 1 token ≈ 4 characters.

In [5]:
import re

def split_into_chunks(text: str) -> list[str]:
    """Split section text into phase chunks using phase headers as boundaries."""
    pattern = r"(?=(?:Preparatory work|Main work|Reworking)\n)"
    parts = re.split(pattern, text)
    return [p.strip() for p in parts if p.strip()]

chunk_tokens = []
for sec, sec_pages in sections.items():
    combined = "\n".join(p.text for p in sec_pages)
    for chunk in split_into_chunks(combined):
        chunk_tokens.append(len(chunk) // 4)

chunk_tokens.sort()
p = lambda pct: chunk_tokens[int(len(chunk_tokens) * pct / 100)]

print(f"Estimated chunk sizes (tokens, 1 token ≈ 4 chars):")
print(f"  min    : {chunk_tokens[0]}")
print(f"  p25    : {p(25)}")
print(f"  median : {p(50)}")
print(f"  p75    : {p(75)}")
print(f"  p90    : {p(90)}")
print(f"  p99    : {p(99)}")
print(f"  max    : {chunk_tokens[-1]}")
print(f"\nTotal chunks: {len(chunk_tokens)}")

Estimated chunk sizes (tokens, 1 token ≈ 4 chars):
  min    : 3
  p25    : 28
  median : 85
  p75    : 197
  p90    : 330
  p99    : 627
  max    : 1317

Total chunks: 460


## 4. README cross-check

The `README.md` describes the ingestion approach. Validate it against what we actually found in the manual.

| README claim | Actual finding | Status |
|---|---|---|
| "chunked by TOC section" | Correct — section numbers are the primary boundary | ✅ |
| Hierarchy example: `Chassis > Rear Suspension > Removal` | Actual hierarchy: `Chapter (int) > Section (e.g. 9.9) > Phase` | ⚠️ misleading example |
| "Extracted images and tables" | Images: yes, embedded JPEGs per page. Tables: this manual has no traditional tables — torque specs are structured inline text, extracted as `TorqueSpec` objects | ⚠️ inaccurate — no tables |
| Cross-ref example: `"See Body Panel Removal, Chapter 4"` | Cross-reference format in this manual is unverified — `crossref.py` is not yet implemented | ⏳ not yet validated |
| No mention of intra-section phase splitting | 32% of sections (72/224) contain multiple phases and need splitting on `Preparatory work` / `Main work` / `Reworking` headers | ❌ missing from README |

**Action:** Update README to reflect the actual chunk hierarchy and clarify torque spec extraction vs. tables. Phase splitting should be added to the ingestion description. Cross-reference format TBD once `crossref.py` is implemented.